In [ ]:
# ASSIGNMENT 1

In [ ]:
# Exercise 3.4

In [1]:
import sys
print(sys.path)

['C:\\Users\\lenovo\\OneDrive - Politecnico di Milano\\Work_cloud\\DOTTORATO\\Robust Optimization', 'C:\\ModelonImpact-1.8.1\\oct-dist\\install\\Python', 'C:\\Users\\lenovo\\OneDrive - Politecnico di Milano\\Work_cloud\\DOTTORATO\\Robust Optimization', 'C:\\ModelonImpact-1.8.1\\oct-dist\\Python37\\python37.zip', 'C:\\ModelonImpact-1.8.1\\oct-dist\\Python37\\DLLs', 'C:\\ModelonImpact-1.8.1\\oct-dist\\Python37\\lib', 'C:\\ModelonImpact-1.8.1\\oct-dist\\Python37', 'C:\\ModelonImpact-1.8.1\\oct-dist\\myEnv', '', 'C:\\ModelonImpact-1.8.1\\oct-dist\\myEnv\\lib\\site-packages', 'C:\\ModelonImpact-1.8.1\\oct-dist\\myEnv\\lib\\site-packages\\win32', 'C:\\ModelonImpact-1.8.1\\oct-dist\\myEnv\\lib\\site-packages\\win32\\lib', 'C:\\ModelonImpact-1.8.1\\oct-dist\\myEnv\\lib\\site-packages\\Pythonwin', 'C:\\ModelonImpact-1.8.1\\oct-dist\\myEnv\\lib\\site-packages\\IPython\\extensions', 'C:\\Users\\lenovo\\.ipython']


In [2]:
# Add a new path to sys.path
new_path = 'C:\\ModelonImpact-1.8.1\\oct-dist\\Python37\\Lib\\site-packages'
sys.path.append(new_path)

In [4]:
import rsome as rso
import numpy as np
from rsome import ro
from rsome import msk_solver as my_solver  #Import Mosek solver interface
#from rsome import grb_solver as my_solver  #Import Gurobi solver interface

In [121]:
rng = np.random.default_rng(seed=42)

n = 10
m = 20
c = rng.uniform(size=n)
N = 40
pBars = rng.uniform(size=(N,m))
qBar = rng.uniform(size=N)
qBar = qBar/qBar.sum()
d =  rng.uniform(size=m)
zbar = rng.uniform(size=n)
K = 5
A = rng.uniform(size=(K,n))
B = rng.uniform(size=(K,m))
b = 100*rng.uniform(size=K)

Gamma = 0.1
gamma = 10

In [122]:
model = ro.Model('Deterministic_Production_Prob')
#Define variables and uncertain parameters
x = model.dvar(n)
y = model.dvar(m)

#List the objective and constraints
model.max(qBar@(pBars@y)-c@x)
model.st(d@y<= zbar@x)
model.st(A@x+B@y<=b)
model.st(x>=0)
model.st(y>=0)

#Solve the model
model.solve(my_solver)

#Return and print the optimal value and optimal solution
optobj_det = model.get()
xx_det   = x.get()
yy_det   = y.get()


print('The objective of deterministic problem is {0:0.4f}'.format(model.get()))
[xx_det, yy_det]

Being solved by Mosek...
Solution status: optimal
Running time: 0.0293s
The objective of deterministic problem is 39.6859


[array([ 0.        ,  0.        ,  0.        ,  0.        , 26.05926238,
         0.        ,  0.        ,  0.        ,  0.        ,  0.        ]),
 array([ 0.        , 17.38806208,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ,  0.        , 26.65584967,  0.        ,
         0.        ,  0.        ,  0.        ,  0.        , 35.09780721,
         0.        ,  0.        ,  0.        ,  0.        ,  7.33558047])]

In [123]:
model = ro.Model('Raw_RiskAverse_Production_Prob')
#Define variables and uncertain parameters
x = model.dvar(n)
y = model.dvar(m)
s = model.dvar(1)

#Define uncertainty and uncertainty sets
#CODE MISSING HERE
#z = model.rvar(n)
delta = model.rvar(n)
q = model.rvar(N)
ones_array = np.ones(n)
uncertaintySet = (delta.sum()>=-Gamma*(n**0.5),-ones_array<=delta,delta<=ones_array) # Scrivere il vincolo anche con il for loop?
 # I can also avoid defining z, writing the set only in terms of theta and then write after z = Zbars@theta
profituncertaintySet = (q.sum()==1,gamma**(-1)*qBar<=q,q<=gamma*qBar)


#List the objective and constraints
#I COPIED THE DETERMINISTC MODEL BELOW IN EPIGRAPH FORM FOR YOU TO START FROM
model.max(s)
model.st((s <= q@(pBars@y)-c@x).forall(profituncertaintySet)) #MODIFY THE CONSTRAINT TO GET THE ROBUST FORM
#model.st((s <= qBar@(pBars@y)-c@x))
model.st((d@y<= delta*zbar@x+zbar@x).forall(uncertaintySet)) #MODIFY THE CONSTRAINT TO GET THE ROBUST FORM
#model.st(d@y<= zbar@x)
model.st(A@x+B@y<=b)
model.st(x>=0)
model.st(y>=0)

#Solve the model
model.solve(my_solver)

#Return and print the optimal value and optimal solution
optobj_raw = model.get()
xx_raw   = x.get()
yy_raw   = y.get()


print('The objective of Raw Robust problem is {0:0.4f}'.format(model.get()))
[xx_raw, yy_raw]

Being solved by Mosek...
Solution status: optimal
Running time: 0.0411s
The objective of Raw Robust problem is 6.3003


[array([1.57484838, 0.        , 1.19590316, 3.10506257, 1.62802367,
        1.3193961 , 2.96748388, 3.66950365, 1.923737  , 2.01562694]),
 array([ 0.68368508,  5.715644  ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ,  0.        ,  0.        , 11.16302154,
         0.        ,  0.        ,  0.        , 15.4087421 , 24.60431373])]

In [127]:
#Reduced form for AgentA constraint
model = ro.Model('Reduced_RiskAverse_Production_Prob')
#Define variables and uncertain parameters
x = model.dvar(n)
y = model.dvar(m)
s = model.dvar(1)

q = model.rvar(N)
profituncertaintySet = (q.sum()==1,gamma**(-1)*qBar<=q,q<=gamma*qBar)


#ADD SOME DUAL VARIABLES HERE
mygamma = model.dvar(1)
mylambdaplus = model.dvar(n)
mylambdaminus = model.dvar(n)

#List the objective and constraints
#I COPIED THE DETERMINISTC MODEL BELOW IN EPIGRAPH FORM FOR YOU TO START FROM
model.max(s)
#model.st((s <= q@(pBars@y)-c@x).forall(profituncertaintySet)) #MODIFY THE CONSTRAINT TO GET THE ROBUST FORM
model.st((s <= qBar@(pBars@y)-c@x))
model.st(Gamma*(n**0.5)*mygamma + mylambdaplus.sum() + mylambdaminus.sum() <= zbar@x -d@y) #MODIFY THE CONSTRAINT TO GET THE ROBUST FORM
for i in np.arange(n):
    model.st(-mygamma - mylambdaplus[i] + mylambdaminus[i] == -zbar[i]*x[i]) #MODIFY THE CONSTRAINT TO GET THE ROBUST FORM
model.st(A@x+B@y<=b)
model.st(x>=0)
model.st(y>=0)
model.st(mygamma>=0)
model.st(mylambdaplus>=0)
model.st(mylambdaminus>=0)

#Solve the model
model.solve(my_solver)

#Return and print the optimal value and optimal solution
optobj_rob = model.get()
xx_red   = x.get()
yy_red   = y.get()


print('The objective of Reduced Robust problem is {0:0.4f}'.format(model.get()))
[xx_red, yy_red]

Being solved by Mosek...
Solution status: optimal
Running time: 0.0146s
The objective of Reduced Robust problem is 18.8516


[array([1.36855495, 0.        , 1.03924874, 2.69832247, 1.41476468,
        1.14656502, 2.57876556, 0.        , 1.67174176, 1.7515948 ]),
 array([ 0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ,  0.        ,  0.        , 55.28968817,
         0.        ,  0.        ,  0.        ,  0.        ,  0.        ])]

In [99]:
#Reduced form for profit uncertainty
model = ro.Model('Reduced_RiskAverse_Production_Prob')
#Define variables and uncertain parameters
x = model.dvar(n)
y = model.dvar(m)
s = model.dvar(1)

#ADD SOME DUAL VARIABLES HERE
mygammaminus = model.dvar(1)
mygammaplus = model.dvar(1)
mylambdaplus = model.dvar(N)
mylambdaminus = model.dvar(N)

#List the objective and constraints
#I COPIED THE DETERMINISTC MODEL BELOW IN EPIGRAPH FORM FOR YOU TO START FROM
model.max(s)
model.st(-mygammaminus + mygammaplus + gamma*qBar@mylambdaplus - gamma**(-1)*qBar@mylambdaminus <= -s -c@x) #MODIFY THE CONSTRAINT TO GET THE ROBUST FORM
for i in np.arange(N):
    model.st(-mygammaminus + mygammaplus + mylambdaplus[i] - mylambdaminus[i] == -pBars[i,:]@y)
model.st(d@y<= zbar@x) #MODIFY THE CONSTRAINT TO GET THE ROBUST FORM
model.st(A@x+B@y<=b)
model.st(x>=0)
model.st(y>=0)
model.st(mygammaplus>=0)
model.st(mygammaminus>=0)
model.st(mylambdaplus>=0)
model.st(mylambdaminus>=0)

#Solve the model
model.solve(my_solver)

#Return and print the optimal value and optimal solution
optobj_rob = model.get()
xx_red   = x.get()
yy_red   = y.get()


print('The objective of Reduced Robust problem is {0:0.4f}'.format(model.get()))
[xx_red, yy_red]

Being solved by Mosek...
Solution status: optimal
Running time: 0.0123s
The objective of Reduced Robust problem is 26.3104


[array([ 0.        ,  0.        ,  0.        ,  0.        , 22.24161017,
         0.        ,  0.        ,  0.        ,  4.24567771,  0.        ]),
 array([11.71651767,  9.47754881,  0.        ,  0.        , 17.92833045,
         0.        ,  0.        ,  0.        ,  4.14257503,  9.24974967,
         0.        ,  0.        ,  0.        ,  0.        ,  6.87573673,
         0.        ,  0.        ,  0.        ,  0.        , 18.22916251])]

In [128]:
# Complete reduced problem
model = ro.Model('Reduced_RiskAverse_Production_Prob')
#Define variables and uncertain parameters
x = model.dvar(n)
y = model.dvar(m)
s = model.dvar(1)

#ADD SOME DUAL VARIABLES HERE
# Variables introduced to cope with profit uncertainty
mygammaminus = model.dvar(1)
mygammaplus = model.dvar(1)
mylambdaplus = model.dvar(N)
mylambdaminus = model.dvar(N)
# Variables introduced to cope with agentA uncertainty
mygamma = model.dvar(1)
mylambdaplusA = model.dvar(n)
mylambdaminusA = model.dvar(n)

#List the objective and constraints
model.max(s)
model.st(A@x+B@y<=b)
# Reformulated constraints for profit uncertainty
model.st(-mygammaminus + mygammaplus + gamma*qBar@mylambdaplus - gamma**(-1)*qBar@mylambdaminus <= -s -c@x) #MODIFY THE CONSTRAINT TO GET THE ROBUST FORM
for i in np.arange(N):
    model.st(-mygammaminus + mygammaplus + mylambdaplus[i] - mylambdaminus[i] == -pBars[i,:]@y)
# Reformulated constraints for agentA uncertainty
model.st(Gamma*(n**0.5)*mygamma + mylambdaplusA.sum() + mylambdaminusA.sum() <= zbar@x -d@y) #MODIFY THE CONSTRAINT TO GET THE ROBUST FORM
for i in np.arange(n):
    model.st(-mygamma - mylambdaplusA[i] + mylambdaminusA[i] == -zbar[i]*x[i])
# Additional constraints
model.st(x>=0)
model.st(y>=0)
model.st(mygammaplus>=0)
model.st(mygammaminus>=0)
model.st(mylambdaplus>=0)
model.st(mylambdaminus>=0)
model.st(mygamma>=0)
model.st(mylambdaplusA>=0)
model.st(mylambdaminusA>=0)

#Solve the model
model.solve(my_solver)

#Return and print the optimal value and optimal solution
optobj_rob = model.get()
xx_red   = x.get()
yy_red   = y.get()


print('The objective of Reduced Robust problem is {0:0.4f}'.format(model.get()))
[xx_red, yy_red]

Being solved by Mosek...
Solution status: optimal
Running time: 0.0257s
The objective of Reduced Robust problem is 6.3003


[array([1.57484838, 0.        , 1.19590316, 3.10506257, 1.62802367,
        1.3193961 , 2.96748388, 3.66950365, 1.923737  , 2.01562694]),
 array([ 0.68368508,  5.715644  ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
         0.        ,  0.        ,  0.        ,  0.        , 11.16302154,
         0.        ,  0.        ,  0.        , 15.4087421 , 24.60431373])]